# 06 · Reproducir la métrica del champion desde el Model Registry

Notebook **limpio**: no entrena nada ni reutiliza objetos de otros notebooks. Carga el modelo publicado como
`models:/ssd-failure-modelo@champion`, lo aplica a la prueba (noviembre de 2019, leída del mismo snapshot verificado por
SHA-256) y comprueba que la PR-AUC por evento coincide con la que quedó registrada en su corrida `fase=final`.

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import mlflow
from mlflow import MlflowClient

sys.path.insert(0, str(Path.cwd().parent))
from src import datos, pipeline as pl

DATOS = Path(os.environ.get("DATOS_DIR", "/datos"))
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
cliente = MlflowClient()
version = cliente.get_model_version_by_alias(pl.NOMBRE_MODELO_REGISTRO, "champion")
corrida = cliente.get_run(version.run_id)
print(f"{pl.NOMBRE_MODELO_REGISTRO}@champion → versión {version.version}, run {version.run_id}")
print("familia:", corrida.data.tags.get("familia"), "· commit:", corrida.data.tags.get("git_commit"),
      "· datos:", corrida.data.params.get("datos_sha256", "")[:16], "…")

ssd-failure-modelo@champion → versión 1, run 74b6bda293ea4c7c82c9e7d221db0762
familia: xgboost · commit: a367744c943917eba328be47ceb9267ed2d47532 · datos: c0b1ea5c055579e0 …


In [2]:
manifiesto = datos.leer_manifiesto(DATOS / "obt_mc1")
assert manifiesto["dataset_sha256"] == corrida.data.params["datos_sha256"], "El snapshot no es el del champion"
prueba = datos.cargar_cache(DATOS / "cache_R", desde=datos.INICIO_PRUEBA, hasta=datos.FIN_PRUEBA,
                            sha_esperado=manifiesto["dataset_sha256"])
modelo = mlflow.sklearn.load_model(f"models:/{pl.NOMBRE_MODELO_REGISTRO}@champion")
columnas = list(modelo.feature_names_in_) if hasattr(modelo, "feature_names_in_") else pl.columnas_modelo(prueba, "R")
score = modelo.predict_proba(prueba[columnas])[:, 1]
curva = pl.curva_evento(prueba["SSD_KEY"], prueba["OBSERVATION_DATE"], prueba["DAYS_TO_FAILURE"], score)
registrada = corrida.data.metrics["test_pr_auc_evento"]
print(f"PR-AUC por evento en la prueba: reproducida {curva.pr_auc:.6f} · registrada {registrada:.6f}")
assert np.isclose(curva.pr_auc, registrada, rtol=1e-6, atol=1e-9), "No se reproduce la métrica registrada"
print("✔ La métrica del champion se reproduce exactamente desde el Model Registry")

/opt/conda/lib/python3.12/site-packages/xgboost/core.py:705: UserWarning: [10:20:29] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1748362148159/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


PR-AUC por evento en la prueba: reproducida 0.007540 · registrada 0.007540
✔ La métrica del champion se reproduce exactamente desde el Model Registry
